In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import anndata as ad
import tensorflow as tf
import wandb
import keras
import crested
import pickle 

%matplotlib inline

2025-08-26 14:21:41.704343: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-26 14:21:46.884889: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756210907.881249  584371 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756210908.154740  584371 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1756210910.948162  584371 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [2]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  0


2025-08-26 14:23:29.667761: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [3]:
genome_file = "/staging/leuven/res_00001/genomes/homo_sapiens/CHM13v2_maskedY_rCRS/fasta/chm13v2.0_maskedY_rCRS.fa"
chromsizes_file = "/staging/leuven/res_00001/genomes/homo_sapiens/CHM13v2_maskedY_rCRS/fasta/chm13v2.0_maskedY_rCRS.chrom.sizes"


# Set the genome
genome = crested.Genome(
    "/staging/leuven/res_00001/genomes/homo_sapiens/CHM13v2_maskedY_rCRS/fasta/chm13v2.0_maskedY_rCRS.fa", 
    "/staging/leuven/res_00001/genomes/homo_sapiens/CHM13v2_maskedY_rCRS/fasta/chm13v2.0_maskedY_rCRS.chrom.sizes"
)

crested.register_genome(
    genome
)

2025-08-26T14:23:36.003442+0200 INFO Genome chm13v2.0_maskedY_rCRS registered.


# SN model

In [4]:
# load adata

adata_path = "/lustre1/project/stg_00090/ASA/analysis/analysis_Olga/3_T2T_analysis/2_CREsted_models/peak_regression/adata_sn_mean_norm_specific.h5ad"
adata_sn = ad.read_h5ad(adata_path )

adata_sn.X.shape

(9, 194202)

In [6]:
adata_sn.obs_names

Index(['Astro', 'DopaN', 'Endo', 'GabaN', 'GlutaN', 'Micro-PVM', 'OPC',
       'Oligo', 'immune'],
      dtype='object')

In [5]:
print(adata_sn.var["split"].value_counts())

split
train    159852
val       19971
test      14379
Name: count, dtype: int64


## Load pretrained model and setup

In [8]:
# First load the pretrained model on all peaks
model_architecture = keras.models.load_model(
    "/lustre1/project/stg_00090/ASA/analysis/analysis_Olga/3_T2T_analysis/2_CREsted_models/deepPeak/deepPeak_SN_mean/checkpoints/21.keras",
    compile=False,  # Choose the basemodel with best validation loss/performance metrics
)

# Use new adata for the datamodule, reduce batch size
datamodule = crested.tl.data.AnnDataModule(
    adata_sn,
    genome=genome_file,
    chromsizes_file=chromsizes_file,
    batch_size=64,
    max_stochastic_shift = 3
)

In [9]:
# Optimizer: default as used for all CREsted models. Make sure that learning rate is lower than it was on the epoch you select the model from.
optimizer = keras.optimizers.Adam(learning_rate=1e-4)
loss = crested.tl.losses.CosineMSELogLoss(max_weight=100) # multiplier=1 to prodict actual values (default - multiplied by 1000)

In [10]:

metrics = [
    keras.metrics.MeanAbsoluteError(),
    keras.metrics.MeanSquaredError(),
    keras.metrics.CosineSimilarity(axis=1),
    crested.tl.metrics.PearsonCorrelation(),
    crested.tl.metrics.ConcordanceCorrelationCoefficient(),
    crested.tl.metrics.PearsonCorrelationLog(),
    crested.tl.metrics.ZeroPenaltyMetric(),
]
        
config = crested.tl.TaskConfig(optimizer, loss, metrics)
print(config)

TaskConfig(optimizer=<keras.src.optimizers.adam.Adam object at 0x14e72c88c510>, loss=<crested.tl.losses._cosinemse_log.CosineMSELogLoss object at 0x14e72c88ec90>, metrics=[<MeanAbsoluteError name=mean_absolute_error>, <MeanSquaredError name=mean_squared_error>, <CosineSimilarity name=cosine_similarity>, <PearsonCorrelation name=pearson_correlation>, <ConcordanceCorrelationCoefficient name=concordance_correlation_coefficient>, <PearsonCorrelationLog name=pearson_correlation_log>, <ZeroPenaltyMetric name=zero_penalty_metric>])


In [11]:
# setup the trainer
trainer = crested.tl.Crested(
    data=datamodule,
    model=model_architecture,
    config=config,
    project_name="deepPeak", # Your wandb project name, not required if you don't need to log performance
    run_name="deepPeak_SN_mean_finetuned", # If using wandb to log, this run's name, as an example
    logger="wandb",
)

## Fit the model

In [12]:
trainer.fit(epochs=60)

wandb: Currently logged in as: olga-sigalova88 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence            │ (None, 2114, 4)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 2114, 512) │     10,240 │ sequence[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 2114, 512) │      2,048 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 2114, 512) │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 2114, 512) │          0 │ activation[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bpnet_1conv         │ (None, 2110, 512) │    786,432 │ dropout[0][0]     │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bpnet_1bn           │ (None, 2110, 512) │      2,048 │ bpnet_1conv[0][0] │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bpnet_1activation   │ (None, 2110, 512) │          0 │ bpnet_1bn[0][0]   │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bpnet_1crop         │ (None, 2110, 512) │          0 │ dropout[0][0]     │
│ (Cropping1D)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 2110, 512) │          0 │ bpnet_1activatio… │
│                     │                   │            │ bpnet_1crop[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bpnet_1dropout      │ (None, 2110, 512) │          0 │ add[0][0]         │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bpnet_2conv         │ (None, 2102, 512) │    786,432 │ bpnet_1dropout[0… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bpnet_2bn           │ (None, 2102, 512) │      2,048 │ bpnet_2conv[0][0] │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bpnet_2activation   │ (None, 2102, 512) │          0 │ bpnet_2bn[0][0]   │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bpnet_2crop         │ (None, 2102, 512) │          0 │ bpnet_1dropout[0… │
│ (Cropping1D)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 2102, 512) │          0 │ bpnet_2activatio… │
│                     │                   │            │ bpnet_2crop[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bpnet_2dropout      │ (None, 2102, 512) │          0 │ add_1[0][0]       │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 6,324,745 (24.13 MB)

 Trainable params: 6,315,529 (24.09 MB)

...

4996/4996 ━━━━━━━━━━━━━━━━━━━━ 0s 229ms/step - concordance_correlation_coefficient: 0.6525 - cosine_similarity: 0.9038 - loss: -0.4498 - mean_absolute_error: 0.0741 - mean_squared_error: 0.0357 - pearson_correlation: 0.6920 - pearson_correlation_log: 0.8254 - zero_penalty_metric: 2.1034

2025-06-09 14:41:38.636797: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_684', 168 bytes spill stores, 168 bytes spill loads



4996/4996 ━━━━━━━━━━━━━━━━━━━━ 1225s 234ms/step - concordance_correlation_coefficient: 0.6525 - cosine_similarity: 0.9038 - loss: -0.4498 - mean_absolute_error: 0.0741 - mean_squared_error: 0.0357 - pearson_correlation: 0.6920 - pearson_correlation_log: 0.8254 - zero_penalty_metric: 2.1034 - val_concordance_correlation_coefficient: 0.6064 - val_cosine_similarity: 0.8965 - val_loss: -0.4161 - val_mean_absolute_error: 0.0801 - val_mean_squared_error: 0.0531 - val_pearson_correlation: 0.6106 - val_pearson_correlation_log: 0.8146 - val_zero_penalty_metric: 1.9916 - learning_rate: 1.0000e-04
Epoch 2/60
4996/4996 ━━━━━━━━━━━━━━━━━━━━ 1148s 230ms/step - concordance_correlation_coefficient: 0.6815 - cosine_similarity: 0.9099 - loss: -0.4771 - mean_absolute_error: 0.0723 - mean_squared_error: 0.0341 - pearson_correlation: 0.7110 - pearson_correlation_log: 0.8349 - zero_penalty_metric: 1.9596 - val_concordance_correlation_coefficient: 0.5995 - val_cosine_similarity: 0.8959 - val_loss: -0.4121 - 

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


batch/batch_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇██
batch/concordance_correlation_coefficient,▁▂▂▄▄▄▅▅▆▆▅▅▅▅▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇██████
batch/cosine_similarity,▁▂▃▃▃▄▄▄▄▆▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇█████████
batch/learning_rate,███████████████████████▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
batch/loss,██▇▆▆▅▅▄▄▄▄▄▄▄▄▄▄▄▃▃▃▃▃▃▃▃▄▂▂▂▂▂▂▁▁▁▁▁▁▁
batch/mean_absolute_error,██▇▇▆▅▅▅▅▅▅▅▅▄▄▅▅▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▂
batch/mean_squared_error,██▇▇▆▆▅▄▄▅▅▄▄▄▄▄▄▄▄▂▃▃▂▂▂▁▁▂▂▂▂▂▂▂▂▁▁▁▁▂
batch/pearson_correlation,▁▃▃▃▄▄▄▄▄▅▅▅▄▅▅▆▅▅▅▅▇▇▇▇▇▇▇▇▇▇█▇▇▇▇▇████
batch/pearson_correlation_log,▁▃▃▃▃▅▅▅▅▅▆▆▆▆▆▆▆▆▆▇▆▆▆▇▇▇██████████████
batch/zero_penalty_metric,██▇▇▄▄▃▄▃▃▃▃▅▅▃▃▂▃▃▃▆▅▂▂▁▃▂▂▂▇▅▂▅▅▄▂▃▂▂▂
epoch/concordance_correlation_coefficient,▁▁▃▃▄▄▄▄▅▅▅▅▇▇▇▇▇▇████


# CC finetuning

In [4]:
# load adata

adata_path = "/lustre1/project/stg_00090/ASA/analysis/analysis_Olga/3_T2T_analysis/2_CREsted_models/peak_regression/adata_cc_mean_norm_specific.h5ad"
adata_cc = ad.read_h5ad(adata_path )

adata_cc.X.shape

(21, 274805)

In [5]:
# First load the pretrained model on all peaks
model_architecture = keras.models.load_model(
    "/lustre1/project/stg_00090/ASA/analysis/analysis_Olga/3_T2T_analysis/2_CREsted_models/deepPeak/deepPeak_CC_mean/checkpoints/35.keras",
    compile=False,  # Choose the basemodel with best validation loss/performance metrics
)

# Use new adata for the datamodule, reduce batch size
datamodule = crested.tl.data.AnnDataModule(
    adata_cc,
    genome=genome_file,
    chromsizes_file=chromsizes_file,
    batch_size=64,
    max_stochastic_shift = 3
)

I0000 00:00:1749551963.336446 3111486 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 79195 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-80GB, pci bus id: 0000:bf:00.0, compute capability: 8.0


In [7]:
# Optimizer: default as used for all CREsted models. Make sure that learning rate is lower than it was on the epoch you select the model from.
optimizer = keras.optimizers.Adam(learning_rate=1e-4)
loss = crested.tl.losses.CosineMSELogLoss(max_weight=100) # multiplier=1 to prodict actual values (default - multiplied by 1000)


metrics = [
    keras.metrics.MeanAbsoluteError(),
    keras.metrics.MeanSquaredError(),
    keras.metrics.CosineSimilarity(axis=1),
    crested.tl.metrics.PearsonCorrelation(),
    crested.tl.metrics.ConcordanceCorrelationCoefficient(),
    crested.tl.metrics.PearsonCorrelationLog(),
    crested.tl.metrics.ZeroPenaltyMetric(),
]
        
config = crested.tl.TaskConfig(optimizer, loss, metrics)
print(config)

# setup the trainer
trainer = crested.tl.Crested(
    data=datamodule,
    model=model_architecture,
    config=config,
    project_name="deepPeak", # Your wandb project name, not required if you don't need to log performance
    run_name="deepPeak_CC_mean_finetuned", # If using wandb to log, this run's name, as an example
    logger="wandb",
)

TaskConfig(optimizer=<keras.src.optimizers.adam.Adam object at 0x151514385b10>, loss=<crested.tl.losses._cosinemse_log.CosineMSELogLoss object at 0x1515143846d0>, metrics=[<MeanAbsoluteError name=mean_absolute_error>, <MeanSquaredError name=mean_squared_error>, <CosineSimilarity name=cosine_similarity>, <PearsonCorrelation name=pearson_correlation>, <ConcordanceCorrelationCoefficient name=concordance_correlation_coefficient>, <PearsonCorrelationLog name=pearson_correlation_log>, <ZeroPenaltyMetric name=zero_penalty_metric>])


In [8]:
trainer.fit(epochs=30)

wandb: Currently logged in as: olga-sigalova88 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence            │ (None, 2114, 4)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 2114, 512) │     10,240 │ sequence[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 2114, 512) │      2,048 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 2114, 512) │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 2114, 512) │          0 │ activation[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bpnet_1conv         │ (None, 2110, 512) │    786,432 │ dropout[0][0]     │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bpnet_1bn           │ (None, 2110, 512) │      2,048 │ bpnet_1conv[0][0] │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bpnet_1activation   │ (None, 2110, 512) │          0 │ bpnet_1bn[0][0]   │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bpnet_1crop         │ (None, 2110, 512) │          0 │ dropout[0][0]     │
│ (Cropping1D)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 2110, 512) │          0 │ bpnet_1activatio… │
│                     │                   │            │ bpnet_1crop[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bpnet_1dropout      │ (None, 2110, 512) │          0 │ add[0][0]         │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bpnet_2conv         │ (None, 2102, 512) │    786,432 │ bpnet_1dropout[0… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bpnet_2bn           │ (None, 2102, 512) │      2,048 │ bpnet_2conv[0][0] │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bpnet_2activation   │ (None, 2102, 512) │          0 │ bpnet_2bn[0][0]   │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bpnet_2crop         │ (None, 2102, 512) │          0 │ bpnet_1dropout[0… │
│ (Cropping1D)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 2102, 512) │          0 │ bpnet_2activatio… │
│                     │                   │            │ bpnet_2crop[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bpnet_2dropout      │ (None, 2102, 512) │          0 │ add_1[0][0]       │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 6,330,901 (24.15 MB)

 Trainable params: 6,321,685 (24.12 MB)

...

6906/6906 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step - concordance_correlation_coefficient: 0.6025 - cosine_similarity: 0.8743 - loss: -0.3843 - mean_absolute_error: 0.0858 - mean_squared_error: 0.0264 - pearson_correlation: 0.6575 - pearson_correlation_log: 0.7621 - zero_penalty_metric: 19.1417

2025-06-10 12:56:51.872314: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_684', 168 bytes spill stores, 168 bytes spill loads

2025-06-10 12:56:52.004848: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_684', 48 bytes spill stores, 48 bytes spill loads

2025-06-10 12:56:52.007087: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_684', 4276 bytes spill stores, 4252 bytes spill loads



6906/6906 ━━━━━━━━━━━━━━━━━━━━ 818s 110ms/step - concordance_correlation_coefficient: 0.6025 - cosine_similarity: 0.8743 - loss: -0.3843 - mean_absolute_error: 0.0858 - mean_squared_error: 0.0264 - pearson_correlation: 0.6575 - pearson_correlation_log: 0.7621 - zero_penalty_metric: 19.1417 - val_concordance_correlation_coefficient: 0.5990 - val_cosine_similarity: 0.8702 - val_loss: -0.3677 - val_mean_absolute_error: 0.0895 - val_mean_squared_error: 0.0311 - val_pearson_correlation: 0.6093 - val_pearson_correlation_log: 0.7530 - val_zero_penalty_metric: 17.8219 - learning_rate: 1.0000e-04
Epoch 2/30
6906/6906 ━━━━━━━━━━━━━━━━━━━━ 725s 105ms/step - concordance_correlation_coefficient: 0.6289 - cosine_similarity: 0.8800 - loss: -0.4035 - mean_absolute_error: 0.0841 - mean_squared_error: 0.0249 - pearson_correlation: 0.6783 - pearson_correlation_log: 0.7701 - zero_penalty_metric: 18.8252 - val_concordance_correlation_coefficient: 0.6059 - val_cosine_similarity: 0.8709 - val_loss: -0.3724 -

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


batch/batch_step,▁▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇███████
batch/concordance_correlation_coefficient,▁▃▃▃▃▃▃▃▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▅▅▅▅▅▅█▅▅▅▅▅▅▅▅▅
batch/cosine_similarity,▁▂▄▄▄▅▅▅▅▅▅▅▅▆▆▅▅▅▆▆▆▆▆▆▆▇▇▇▇█▇▇████████
batch/learning_rate,██████████████████████▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
batch/loss,███▇▇▆▆▆▆▆▆▆▆▅▅▆▅▅▅▄▅▄▄▄▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▂
batch/mean_absolute_error,█▆▆▆▆▆▅▅▅▆▆▅▅▅▅▄▄▄▄▄▄▄▃▃▃▂▂▁▂▂▂▂▃▂▂▂▁▁▂▂
batch/mean_squared_error,█▇▅▆▆▆▅▅▄▄▄▄▄▄▄▄▄▄▄▄▄▂▂▂▂▂▂▂▂▂▂▁▂▂▂▁▁▁▁▁
batch/pearson_correlation,▁▁▁▁▄▃▃▃▅▄▄▅▅▅▄▅▅▅▅▅▆▆▆▅▅▇▇▇▇▇▇▇██████▇█
batch/pearson_correlation_log,▁▁▂▂▂▃▅▄▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇█▇██████████
batch/zero_penalty_metric,█▇▆▆▆▅▅▆▅▅▅▅▅▅▅▃▅▅▅▅▁▅▅▅▅▅▅▅▃▃▄▃▃▄▄▄▃▃▄▄
epoch/concordance_correlation_coefficient,▁▁▃▃▃▃▄▄▅▅▅▅▅▅▇▇▇▇██████
